# Getting the Silver Data and Pushing to Gold

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df = spark.read.table("ecom.silver.fact_silver_orders")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
table_name = "ecom.gold.fact_gold_orders"
silver_df = df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "approval_time_hours",
    "delivery_time_days",
    "delivery_delay_days",
    "is_delivered_on_time",
    "approved_validation",
    "delivery_validation",
    "order_year_month",
    "ingest_date",
    "file_date",
    "created_date",
    "updated_date",
    "data_source"
)
 

In [0]:

if spark.catalog.tableExists(table_name):

    gold_delta = DeltaTable.forName(spark, table_name)

    (
        gold_delta.alias("target")
        .merge(
            silver_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Rows Appended")

else:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )